# 04 — Execução dos treinamentos

Este notebook executa os treinamentos dos cenários que exigem adaptação do modelo:

```text
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

Os cenários `C0` e `C1` não são treinados neste notebook. Eles usam o modelo base diretamente e serão tratados no notebook de avaliação.

A função deste notebook é transformar o plano documentado no `03_training_documentation.ipynb` em artefatos treinados, salvando os adaptadores em diretórios separados e registrando um manifesto final da execução.

## 0. Instalar dependências de treinamento

Antes de executar o treinamento dos adaptadores, é necessário instalar as dependências específicas da etapa de treinamento. Essas dependências ficam registradas no arquivo:

```text
requirements-train.txt
```

Esse arquivo inclui bibliotecas do ecossistema Hugging Face usadas para carregar modelos, aplicar LoRA/QLoRA, executar fine-tuning supervisionado e treinar com otimização por preferência.

Esta etapa deve ser executada dentro do kernel correto:

```text
Python (pi-defense-exp)
```

O uso do kernel correto é importante porque as dependências precisam ser instaladas no mesmo ambiente virtual usado pelos notebooks. Caso contrário, o pacote pode ser instalado em outro Python e continuar indisponível durante a execução do treinamento.

No ambiente RunPod, é comum que o `torch` já venha instalado na imagem base com suporte à versão correta de CUDA. Por isso, o `requirements-train.txt` deve evitar reinstalar `torch` diretamente, a menos que isso tenha sido decidido explicitamente para a imagem usada. Reinstalar PyTorch manualmente pode causar incompatibilidades com CUDA ou com os drivers da GPU.

Esta célula deve ser executada uma vez antes das etapas de treinamento. Se as dependências já estiverem instaladas, o `pip` apenas confirmará que os pacotes já estão satisfeitos.


In [1]:
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path("/workspace/pi-defense-exp")
REQUIREMENTS_TRAIN = PROJECT_ROOT / "requirements-train.txt"

if not REQUIREMENTS_TRAIN.exists():
    raise FileNotFoundError(
        f"Arquivo de dependências de treinamento não encontrado: {REQUIREMENTS_TRAIN}"
    )

print("Python usado para instalação:", sys.executable)
print("Arquivo de dependências:", REQUIREMENTS_TRAIN)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REQUIREMENTS_TRAIN),
    ],
    check=True,
)

print("Dependências de treinamento instaladas/verificadas com sucesso.")

Python usado para instalação: /workspace/pi-defense-exp/.venv/bin/python
Arquivo de dependências: /workspace/pi-defense-exp/requirements-train.txt
Dependências de treinamento instaladas/verificadas com sucesso.


In [2]:
import importlib.util
from importlib.metadata import version, PackageNotFoundError

import pandas as pd

packages_to_check = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("trl", "trl"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface-hub"),
    ("safetensors", "safetensors"),
    ("sentencepiece", "sentencepiece"),
]

package_rows = []

for import_name, dist_name in packages_to_check:
    try:
        pkg_version = version(dist_name)
        installed = True
    except PackageNotFoundError:
        pkg_version = None
        installed = False

    import_available = importlib.util.find_spec(import_name) is not None

    if installed and import_available:
        status = "ok"
    elif installed and not import_available:
        status = "installed_but_import_not_found"
    elif not installed and import_available:
        status = "import_found_but_distribution_not_found"
    else:
        status = "missing"

    package_rows.append({
        "package": import_name,
        "distribution": dist_name,
        "version": pkg_version,
        "installed": installed,
        "import_available": import_available,
        "status": status,
    })

package_check_df = pd.DataFrame(package_rows)
display(package_check_df)

if not package_check_df["import_available"].all():
    missing_imports = package_check_df.loc[
        ~package_check_df["import_available"],
        "package",
    ].tolist()

    raise RuntimeError(
        "Algumas dependências obrigatórias de treinamento não estão disponíveis para import: "
        + ", ".join(missing_imports)
    )

,package,distribution,version,installed,import_available,status
0,torch,torch,2.8.0+cu128,True,True,ok
1,transformers,transformers,5.12.1,True,True,ok
2,accelerate,accelerate,1.14.0,True,True,ok
3,peft,peft,0.19.1,True,True,ok
4,trl,trl,1.7.0,True,True,ok
5,datasets,datasets,5.0.0,True,True,ok
6,huggingface_hub,huggingface-hub,1.21.0,True,True,ok
7,safetensors,safetensors,0.8.0,True,True,ok
8,sentencepiece,sentencepiece,0.2.1,True,True,ok


In [3]:
import torch

print("Torch version:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    raise RuntimeError(
        "CUDA não está disponível. O treinamento com Llama 3.1 8B provavelmente não será viável sem GPU."
    )

Torch version: 2.8.0+cu128
CUDA disponível: True
GPU: NVIDIA GeForce RTX 5090
CUDA version: 12.8


## 1. Objetivo do notebook

Este notebook tem como objetivo executar o treinamento dos adaptadores usados nos cenários defendidos do experimento.

O desenho experimental considera cinco cenários:

```text
C0 — Base model
C1 — StruQ format-only
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

Neste notebook, apenas os cenários `C2`, `C3` e `C4` são treinados. A razão é que `C0` corresponde ao modelo base sem defesa e `C1` corresponde ao mesmo modelo base avaliado com prompt estruturado, sem ajuste adicional.

Os treinamentos seguem a regra metodológica definida nos notebooks anteriores:

```text
Mesmo modelo base para todos os cenários.
Mesma base canônica de dados.
Views diferentes apenas conforme o mecanismo de defesa.
Adaptadores separados para C2, C3 e C4.
Mesmos arquivos finais de avaliação para todos os cenários.
```

A execução será feita usando o modelo base:

```text
meta-llama/Llama-3.1-8B-Instruct
```

Como esse modelo possui aproximadamente 8 bilhões de parâmetros, este notebook usa uma estratégia baseada em adaptadores LoRA/QLoRA, em vez de fine-tuning completo. Isso reduz o custo computacional e torna a execução mais viável em uma GPU no RunPod.

Ao final deste notebook, devem ser produzidos:

```text
adapters/struq/
adapters/secalign/
adapters/ih/
logs/training/
manifests/training/04_run_training_manifest.json
manifests/training/04_run_training_manifest.md
```

## 2. Validação do ambiente e imports

Nesta etapa, o notebook valida se está sendo executado com o kernel correto e importa as bibliotecas necessárias para treinamento.

A validação do kernel é importante porque o treinamento depende de bibliotecas instaladas no ambiente virtual do projeto, como `transformers`, `datasets`, `peft`, `trl`, `accelerate` e `bitsandbytes`.

O Python esperado é:

```text
/workspace/pi-defense-exp/.venv/bin/python
```

O kernel esperado é:

```text
Python (pi-defense-exp)
```

Se o notebook estiver usando outro interpretador, a execução será interrompida antes de carregar modelos ou iniciar treinamento. Isso evita salvar adaptadores, logs ou manifestos usando um ambiente incorreto.

In [4]:
import os
import sys
import json
import yaml
import time
import platform
import random
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login, whoami
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, DPOTrainer, SFTConfig, DPOConfig

PROJECT_ROOT = Path("/workspace/pi-defense-exp")
EXPECTED_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
CURRENT_PYTHON = Path(sys.executable)
EXPECTED_KERNEL = "Python (pi-defense-exp)"

print("Python atual:", CURRENT_PYTHON)
print("Python esperado:", EXPECTED_PYTHON)

if CURRENT_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        "Este notebook deve ser executado com o kernel 'Python (pi-defense-exp)'. "
        "Selecione esse kernel no Jupyter antes de continuar."
    )

print("Validação inicial concluída.")
print("Kernel correto para o notebook 04.")

Python atual: /workspace/pi-defense-exp/.venv/bin/python
Python esperado: /workspace/pi-defense-exp/.venv/bin/python
Validação inicial concluída.
Kernel correto para o notebook 04.


## 3. Diretórios, cache e configurações globais

Nesta etapa, serão definidos os diretórios usados pelo notebook de treinamento.

Os diretórios seguem a estrutura criada no notebook `01_environment_setup.ipynb` e reutilizada nos notebooks seguintes. Os dados de treino vêm das views geradas no notebook `02_dataset_creation.ipynb`, enquanto o plano de treinamento vem do notebook `03_training_documentation.ipynb`.

Os principais diretórios usados aqui são:

```text
data/views/
adapters/
logs/training/
manifests/training/
configs/
```

Também serão configurados os caches do Hugging Face dentro da pasta do projeto. Isso evita que modelos e datasets sejam baixados para locais globais do sistema e facilita o controle dos artefatos no ambiente RunPod.

Como o modelo base é grande, manter os caches organizados é importante para depuração, reexecução e eventual limpeza de espaço.

In [5]:
CONFIG_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
VIEWS_DIR = DATA_DIR / "views"
CACHE_DIR = DATA_DIR / "cache"
ADAPTERS_DIR = PROJECT_ROOT / "adapters"
LOG_DIR = PROJECT_ROOT / "logs" / "training"
MANIFEST_DIR = PROJECT_ROOT / "manifests" / "training"
RESULTS_DIR = PROJECT_ROOT / "results"

HF_HOME = CACHE_DIR
HF_HUB_CACHE = CACHE_DIR / "hub"
HF_DATASETS_CACHE = CACHE_DIR / "datasets"

for path in [
    CONFIG_DIR,
    VIEWS_DIR,
    CACHE_DIR,
    ADAPTERS_DIR,
    LOG_DIR,
    MANIFEST_DIR,
    RESULTS_DIR,
    HF_HOME,
    HF_HUB_CACHE,
    HF_DATASETS_CACHE,
]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

print("Project root:", PROJECT_ROOT)
print("Views dir:", VIEWS_DIR)
print("Adapters dir:", ADAPTERS_DIR)
print("Training logs dir:", LOG_DIR)
print("Training manifests dir:", MANIFEST_DIR)
print("HF cache dir:", CACHE_DIR)

Project root: /workspace/pi-defense-exp
Views dir: /workspace/pi-defense-exp/data/views
Adapters dir: /workspace/pi-defense-exp/adapters
Training logs dir: /workspace/pi-defense-exp/logs/training
Training manifests dir: /workspace/pi-defense-exp/manifests/training
HF cache dir: /workspace/pi-defense-exp/data/cache


### 3.1. Seeds experimentais

Nesta etapa, o notebook separa duas ideias diferentes de seed.

A primeira é a seed usada para a construção do dataset:

```text
DATASET_SEED = 42
```

Essa seed não representa uma repetição de treinamento. Ela identifica a base experimental criada anteriormente, ou seja, os splits, amostras e ataques gerados no notebook `02_dataset_creation.ipynb`. Manter essa seed fixa é importante para que todos os cenários sejam treinados e avaliados sobre exatamente a mesma base.

A segunda configuração é a lista de seeds experimentais usadas no treinamento:

```text
EXPERIMENT_SEEDS = [42, 123, 2026]
```

Essas seeds representam réplicas experimentais. Cada cenário treinável será executado uma vez para cada seed, produzindo adaptadores separados. Isso reduz a dependência dos resultados em uma única inicialização, ordem de batches ou variação do processo de treinamento.

Neste notebook, isso significa que os cenários abaixo serão treinados em três réplicas:

```text
C2 — StruQ-like SFT: seeds 42, 123 e 2026
C3 — SecAlign-like DPO: seeds 42, 123 e 2026
C4 — Instruction-Hierarchy-like SFT: seeds 42, 123 e 2026
```

Ao final, o notebook poderá produzir até nove adaptadores:

```text
3 cenários treináveis × 3 seeds experimentais = 9 execuções de treinamento
```

Os cenários C0 e C1 não aparecem nesta etapa porque não treinam adaptadores. Eles serão avaliados posteriormente a partir do modelo base, com ou sem formatação defensiva.

Cada adaptador treinado será salvo em um diretório que inclui o cenário e a seed. Essa organização evita sobrescrita de resultados e torna possível calcular médias, desvios padrão e intervalos de confiança no notebook de avaliação.


In [6]:
DATASET_SEED = 42
EXPERIMENT_SEEDS = [42, 123, 2026]

def set_experiment_seed(seed: int) -> None:
    """Configura a seed usada em uma réplica de treinamento."""
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    set_seed(seed)

    # Mantemos benchmark habilitado para viabilidade computacional.
    # A seed controla inicialização, embaralhamento e amostragem,
    # mas pequenas variações ainda podem ocorrer dependendo dos kernels CUDA.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

    print(f"Seed experimental configurada: {seed}")

print("Dataset seed:", DATASET_SEED)
print("Experiment seeds:", EXPERIMENT_SEEDS)


Dataset seed: 42
Experiment seeds: [42, 123, 2026]


## 4. Carregar configuração e manifestos anteriores

Nesta etapa, o notebook carrega os artefatos produzidos nos notebooks anteriores.

O arquivo `configs/experiment.yaml` contém decisões globais do experimento, como seed, tasks, ataques e tamanhos dos splits.

O manifesto `manifests/data/02_dataset_creation_manifest.json` documenta os arquivos de dados e views criados no notebook de dataset.

O arquivo `configs/training_plan.yaml`, produzido no notebook `03_training_documentation.ipynb`, documenta o plano de treinamento esperado para C2, C3 e C4.

Esses arquivos são carregados para evitar redefinir manualmente decisões experimentais neste notebook. Assim, o treinamento continua consistente com os dados e o plano já documentados.

In [7]:
EXPERIMENT_CONFIG_PATH = CONFIG_DIR / "experiment.yaml"
DATASET_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "data" / "02_dataset_creation_manifest.json"
TRAINING_PLAN_PATH = CONFIG_DIR / "training_plan.yaml"

if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Configuração do experimento não encontrada: {EXPERIMENT_CONFIG_PATH}")

if not DATASET_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifesto do dataset não encontrado: {DATASET_MANIFEST_PATH}")

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as f:
    experiment_config = yaml.safe_load(f)

with open(DATASET_MANIFEST_PATH, "r", encoding="utf-8") as f:
    dataset_manifest = json.load(f)

if TRAINING_PLAN_PATH.exists():
    with open(TRAINING_PLAN_PATH, "r", encoding="utf-8") as f:
        training_plan = yaml.safe_load(f)
else:
    training_plan = {}

print("Configuração do experimento carregada:", EXPERIMENT_CONFIG_PATH)
print("Manifesto do dataset carregado:", DATASET_MANIFEST_PATH)
print("Plano de treinamento carregado:", TRAINING_PLAN_PATH if TRAINING_PLAN_PATH.exists() else "não encontrado")

Configuração do experimento carregada: /workspace/pi-defense-exp/configs/experiment.yaml
Manifesto do dataset carregado: /workspace/pi-defense-exp/manifests/data/02_dataset_creation_manifest.json
Plano de treinamento carregado: /workspace/pi-defense-exp/configs/training_plan.yaml


## 5. Verificar autenticação no Hugging Face

Nesta etapa, será verificado se o ambiente está autenticado no Hugging Face.

Essa verificação é necessária porque o modelo base escolhido é:

```text
meta-llama/Llama-3.1-8B-Instruct
```

Esse checkpoint possui acesso controlado no Hugging Face. Portanto, o ambiente precisa estar autenticado com uma conta que tenha permissão de acesso ao modelo. Sem isso, o carregamento do tokenizer, da configuração ou dos pesos do modelo pode falhar.

A célula abaixo primeiro tenta identificar se já existe uma autenticação válida. Caso não exista, ela solicita um token usando `getpass`, evitando que o token apareça no notebook.

O token não deve ser salvo manualmente em células, logs, manifestos ou commits.

In [8]:
try:
    hf_user = whoami()
    print("Hugging Face login detectado.")
    print("User:", hf_user.get("name"))
except Exception:
    hf_token = getpass("Cole seu token do Hugging Face: ")
    login(token=hf_token, add_to_git_credential=False)
    hf_user = whoami()
    print("Login realizado para o ambiente atual do notebook.")
    print("User:", hf_user.get("name"))

Hugging Face login detectado.
User: leinha


## 6. Verificar GPU e precisão numérica

Nesta etapa, o notebook verifica se existe uma GPU disponível e define a precisão numérica usada no treinamento.

Como o modelo base possui aproximadamente 8 bilhões de parâmetros, o treinamento sem GPU não é viável neste experimento. A execução esperada é em um ambiente RunPod com GPU.

O notebook usará `bf16` quando a GPU oferecer suporte adequado. Caso contrário, usará `fp16`. Essa escolha afeta a configuração de treinamento e o tipo de computação usado na quantização de 4 bits.

A verificação também registra informações úteis para o manifesto final, como nome da GPU, memória disponível e versão do CUDA detectada pelo PyTorch.

In [9]:
if not torch.cuda.is_available():
    raise RuntimeError("Nenhuma GPU CUDA foi detectada. Este notebook espera execução em RunPod com GPU.")

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

bf16_supported = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16
precision_name = "bf16" if bf16_supported else "fp16"

print("GPU:", device_name)
print("CUDA capability:", capability)
print(f"Memória total aproximada: {total_memory_gb:.2f} GB")
print("CUDA disponível no PyTorch:", torch.version.cuda)
print("Precisão escolhida:", precision_name)

GPU: NVIDIA GeForce RTX 5090
CUDA capability: (12, 0)
Memória total aproximada: 31.37 GB
CUDA disponível no PyTorch: 12.8
Precisão escolhida: bf16


## 7. Arquivos de treino e validação

Nesta etapa, serão definidos e validados os arquivos usados por cada cenário com treinamento.

Os arquivos foram produzidos no notebook `02_dataset_creation.ipynb`.

Para SFT, são usados exemplos limpos e atacados vistos:

```text
C2 — StruQ-like SFT
C4 — Instruction-Hierarchy-like SFT
```

Para DPO, são usados apenas exemplos atacados vistos, porque cada exemplo precisa de `chosen` e `rejected`:

```text
C3 — SecAlign-like DPO
```

Os caminhos esperados são:

```text
data/views/struq/train_sft.jsonl
data/views/struq/validation_sft.jsonl

data/views/secalign/train_dpo.jsonl
data/views/secalign/validation_dpo.jsonl

data/views/ih/train_sft.jsonl
data/views/ih/validation_sft.jsonl
```

Antes de iniciar qualquer treinamento, o notebook confere se esses arquivos existem e se possuem as contagens esperadas.

In [10]:
TRAINING_FILES = {
    "c2_struq_sft": {
        "train": VIEWS_DIR / "struq" / "train_sft.jsonl",
        "validation": VIEWS_DIR / "struq" / "validation_sft.jsonl",
        "expected_train_rows": 4200,
        "expected_validation_rows": 700,
        "method": "sft",
    },
    "c3_secalign_dpo": {
        "train": VIEWS_DIR / "secalign" / "train_dpo.jsonl",
        "validation": VIEWS_DIR / "secalign" / "validation_dpo.jsonl",
        "expected_train_rows": 2100,
        "expected_validation_rows": 350,
        "method": "dpo",
    },
    "c4_ih_sft": {
        "train": VIEWS_DIR / "ih" / "train_sft.jsonl",
        "validation": VIEWS_DIR / "ih" / "validation_sft.jsonl",
        "expected_train_rows": 4200,
        "expected_validation_rows": 700,
        "method": "sft",
    },
}

def count_jsonl_lines(path: Path) -> int:
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

file_rows = []
for scenario, info in TRAINING_FILES.items():
    for split_name in ["train", "validation"]:
        path = info[split_name]
        if not path.exists():
            raise FileNotFoundError(f"Arquivo não encontrado para {scenario}/{split_name}: {path}")
        observed = count_jsonl_lines(path)
        expected = info[f"expected_{split_name}_rows"]
        if observed != expected:
            raise RuntimeError(
                f"Contagem inesperada em {scenario}/{split_name}: esperado={expected}, observado={observed}"
            )
        file_rows.append({
            "scenario": scenario,
            "method": info["method"],
            "split": split_name,
            "path": str(path),
            "rows": observed,
        })

training_files_df = pd.DataFrame(file_rows)
display(training_files_df)

,scenario,method,split,path,rows
0,c2_struq_sft,sft,train,/workspace/pi-defense-exp/data/views/struq/tra...,4200
1,c2_struq_sft,sft,validation,/workspace/pi-defense-exp/data/views/struq/val...,700
2,c3_secalign_dpo,dpo,train,/workspace/pi-defense-exp/data/views/secalign/...,2100
3,c3_secalign_dpo,dpo,validation,/workspace/pi-defense-exp/data/views/secalign/...,350
4,c4_ih_sft,sft,train,/workspace/pi-defense-exp/data/views/ih/train_...,4200
5,c4_ih_sft,sft,validation,/workspace/pi-defense-exp/data/views/ih/valida...,700


## 8. Modelo base

Nesta etapa, será definido o modelo base usado pelos cenários experimentais.

A decisão metodológica central é que todos os cenários devem partir do mesmo checkpoint base. Essa escolha reduz fatores de confusão e torna a comparação entre os cenários mais controlada.

O modelo base escolhido é:

```text
meta-llama/Llama-3.1-8B-Instruct
```

Esse modelo será usado diretamente nos cenários sem treinamento:

```text
C0 — Base model
C1 — StruQ format-only
```

E também será o ponto de partida dos adaptadores treinados neste notebook:

```text
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

A regra é:

```text
Mesmo modelo base para todos os cenários.
Adaptadores diferentes apenas para C2, C3 e C4.
Mesmos arquivos de avaliação para todos os cenários.
```

Como o modelo possui acesso controlado no Hugging Face, o ambiente precisa estar autenticado antes do carregamento. Além disso, por ser um modelo de 8B parâmetros, o notebook usa quantização em 4 bits e LoRA/QLoRA para reduzir o uso de memória.

In [11]:
BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

model_plan = {
    "base_model_id": BASE_MODEL_ID,
    "model_family": "causal_lm",
    "uses_chat_template": True,
    "requires_huggingface_login": True,
    "same_base_model_for_all_scenarios": True,
    "adapter_strategy": "LoRA/QLoRA",
}

model_plan

{'base_model_id': 'meta-llama/Llama-3.1-8B-Instruct',
 'model_family': 'causal_lm',
 'uses_chat_template': True,
 'requires_huggingface_login': True,
 'same_base_model_for_all_scenarios': True,
 'adapter_strategy': 'LoRA/QLoRA'}

## 9. Configurações LoRA/QLoRA

Nesta etapa, serão definidas as configurações de LoRA/QLoRA usadas nos treinamentos.

O objetivo é treinar adaptadores pequenos em vez de atualizar todos os pesos do modelo base. Isso reduz significativamente o custo computacional e permite executar o experimento em uma única GPU, dependendo da memória disponível.

A quantização em 4 bits será usada para carregar o modelo base com menor consumo de memória. O LoRA será aplicado aos principais módulos lineares da arquitetura Llama, como projeções de atenção e camadas de MLP.

A configuração definida aqui será compartilhada pelos cenários C2, C3 e C4 para manter a comparação mais controlada. O que muda entre os cenários é a view de treinamento e o algoritmo utilizado, não o modelo base nem a estratégia geral de adaptação.

In [12]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

lora_plan = {
    "r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "target_modules": lora_config.target_modules,
    "load_in_4bit": True,
    "quant_type": "nf4",
    "compute_dtype": precision_name,
}

lora_plan

{'r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'target_modules': {'down_proj',
  'gate_proj',
  'k_proj',
  'o_proj',
  'q_proj',
  'up_proj',
  'v_proj'},
 'load_in_4bit': True,
 'quant_type': 'nf4',
 'compute_dtype': 'bf16'}

## 10. Preparação dos datasets para treinamento

Nesta etapa, os arquivos JSONL serão carregados como `Dataset` do Hugging Face e preparados no formato esperado por cada tipo de treinamento.

Para SFT, o notebook transforma cada exemplo em um campo `text`, contendo uma conversa com uma mensagem de usuário e uma resposta do assistente. Essa conversa é renderizada com o `chat_template` do tokenizer do modelo base.

Para DPO, o notebook mantém o formato de preferência:

```text
prompt
chosen
rejected
```

Esse formato é necessário porque o DPO compara uma resposta preferida com uma resposta rejeitada para o mesmo prompt. A documentação do TRL descreve que exemplos de DPO devem conter prompt, chosen e rejected. 

O objetivo desta etapa é preparar os dados sem alterar a semântica definida no notebook 02. As respostas corretas continuam vindo de `expected_answer`, e as respostas rejeitadas continuam vindo de `attack_target`.

In [13]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def build_sft_text(example: dict, tokenizer) -> str:
    if "prompt" in example and "completion" in example:
        user_content = example["prompt"]
        assistant_content = example["completion"]
    elif "messages" in example and "completion" in example:
        messages = list(example["messages"])
        messages.append({"role": "assistant", "content": example["completion"]})
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    else:
        raise ValueError(f"Formato SFT desconhecido no exemplo: {example.keys()}")

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def load_sft_dataset(path: Path, tokenizer) -> Dataset:
    rows = read_jsonl(path)
    prepared_rows = []

    for row in rows:
        prepared_rows.append({
            "text": build_sft_text(row, tokenizer),
            "source_id": row.get("source_id"),
            "task_name": row.get("task_name"),
            "attack_type": row.get("attack_type", "clean"),
        })

    return Dataset.from_list(prepared_rows)


def load_dpo_dataset(path: Path) -> Dataset:
    rows = read_jsonl(path)
    prepared_rows = []

    for row in rows:
        prepared_rows.append({
            "prompt": row["prompt"],
            "chosen": row["chosen"],
            "rejected": row["rejected"],
            "source_id": row.get("source_id"),
            "task_name": row.get("task_name"),
            "attack_type": row.get("attack_type"),
        })

    return Dataset.from_list(prepared_rows)

## 11. Carregar tokenizer

Nesta etapa, será carregado o tokenizer do modelo base.

O tokenizer é necessário antes da preparação final dos datasets SFT, porque os exemplos SFT serão convertidos para texto usando o `chat_template` do próprio modelo.

Também é importante garantir que o tokenizer tenha um token de padding definido. Em modelos causais, é comum usar o token de fim de sequência como token de padding quando o tokenizer não possui um `pad_token` específico.

In [14]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    cache_dir=str(HF_HUB_CACHE),
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer carregado:", BASE_MODEL_ID)
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)
print("chat_template disponível:", tokenizer.chat_template is not None)

Tokenizer carregado: meta-llama/Llama-3.1-8B-Instruct
pad_token: <|eot_id|>
eos_token: <|eot_id|>
chat_template disponível: True


## 12. Funções de carregamento do modelo

Nesta etapa, será definida uma função para carregar o modelo base em 4 bits.

Cada cenário será treinado a partir do mesmo checkpoint base, mas em execuções separadas. Isso evita que um treinamento continue acidentalmente a partir dos pesos adaptados de outro cenário.

A função abaixo carrega o modelo, prepara-o para treinamento em k-bit e desativa o cache durante o treinamento. Essa configuração é comum em treinamento com gradient checkpointing e adaptadores.

In [15]:
def load_base_model_for_training():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        cache_dir=str(HF_HUB_CACHE),
        quantization_config=quantization_config,
        device_map="auto",
        dtype=compute_dtype,
    )

    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    return model

## 13. Parâmetros de treinamento

Nesta etapa, serão definidos os parâmetros de treinamento usados nos cenários.

Os valores abaixo são intencionalmente conservadores para uma primeira execução em RunPod com uma GPU. O objetivo é validar o pipeline completo e produzir adaptadores iniciais, sem transformar esta etapa em um treinamento longo demais.

A lista `SCENARIOS_TO_RUN` controla quais cenários serão treinados nesta execução. As seeds não devem ser removidas dessa etapa: cada cenário listado será treinado uma vez para cada valor em `EXPERIMENT_SEEDS`.

Assim, se a lista de cenários contiver:

```text
c2_struq_sft
c3_secalign_dpo
c4_ih_sft
```

e as seeds forem:

```text
42, 123, 2026
```

o notebook executará nove treinamentos de adaptadores, um para cada combinação entre cenário e seed.

A recomendação inicial, caso seja necessário testar o pipeline com menor custo, é reduzir temporariamente `SCENARIOS_TO_RUN` para apenas um cenário. No entanto, a lista `EXPERIMENT_SEEDS` deve continuar registrada no notebook e no manifesto, porque ela faz parte do desenho experimental completo.

Depois que o pipeline estiver validado, é possível ajustar hiperparâmetros como número de épocas, learning rate, batch size efetivo, `max_length` e frequência de salvamento.


In [16]:
# Edite esta lista para controlar quais cenários serão treinados nesta execução.
SCENARIOS_TO_RUN = [
    "c2_struq_sft",
    "c3_secalign_dpo",
    "c4_ih_sft",
]

MAX_SEQ_LENGTH = 1024
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LOGGING_STEPS = 10
SAVE_STEPS = 100
EVAL_STEPS = 100

SFT_LEARNING_RATE = 2e-4
DPO_LEARNING_RATE = 5e-5

training_hyperparameters = {
    "dataset_seed": DATASET_SEED,
    "experiment_seeds": EXPERIMENT_SEEDS,
    "replicates_per_trained_scenario": len(EXPERIMENT_SEEDS),
    "scenarios_to_run": SCENARIOS_TO_RUN,
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "sft_learning_rate": SFT_LEARNING_RATE,
    "dpo_learning_rate": DPO_LEARNING_RATE,
    "precision": precision_name,
}

training_hyperparameters

{'dataset_seed': 42,
 'experiment_seeds': [42, 123, 2026],
 'replicates_per_trained_scenario': 3,
 'scenarios_to_run': ['c2_struq_sft', 'c3_secalign_dpo', 'c4_ih_sft'],
 'max_seq_length': 1024,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'per_device_eval_batch_size': 1,
 'gradient_accumulation_steps': 16,
 'sft_learning_rate': 0.0002,
 'dpo_learning_rate': 5e-05,
 'precision': 'bf16'}

### 13.1. Logs incrementais de treinamento

Antes de iniciar os treinamentos, o notebook cria uma estrutura de logs incrementais para registrar cada run separadamente.

Essa etapa é importante porque o notebook executa múltiplas combinações de cenário e seed. No desenho atual, são três cenários treináveis:

```text
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

E cada um deles é executado com três seeds experimentais:

```text
42, 123, 2026
```

Isso resulta em nove runs de treinamento. Como essas execuções podem ser longas e consumir bastante memória de GPU, não é seguro depender apenas de um log final gerado depois que tudo termina. Se uma execução for interrompida no meio, por erro de CUDA, falta de memória, falha de autenticação, reinício do ambiente ou interrupção manual, um log criado apenas no final pode nunca ser salvo.

Por isso, este notebook passa a registrar eventos de forma incremental. Cada run terá eventos de início, conclusão ou falha gravados imediatamente em um arquivo global:

```text
logs/training/04_run_training_events.jsonl
```

Esse arquivo usa o formato JSONL, em que cada linha é um evento independente. Isso facilita inspeção manual, leitura programática e recuperação parcial dos resultados.

Além do log global, cada combinação de cenário e seed terá um diretório próprio de logs:

```text
logs/training/runs/<scenario_name>/seed_<seed>/
```

Por exemplo:

```text
logs/training/runs/c2_struq_sft/seed_42/
logs/training/runs/c3_secalign_dpo/seed_123/
logs/training/runs/c4_ih_sft/seed_2026/
```

Dentro desses diretórios, o notebook pode salvar metadados da run, métricas retornadas pelo trainer e, em caso de erro, um arquivo `error.txt` com o traceback completo. Essa organização torna o treinamento mais auditável e permite saber exatamente quais réplicas foram concluídas e quais precisam ser reexecutadas.

O log final e o manifesto continuam existindo, mas eles passam a ser complementares aos logs incrementais. O log incremental é o registro mais seguro durante a execução; o manifesto final é a consolidação produzida ao final do notebook.

In [17]:
import traceback

RUNS_LOG_DIR = LOG_DIR / "runs"
EVENTS_LOG_PATH = LOG_DIR / "04_run_training_events.jsonl"
SUMMARY_LOG_JSON_PATH = LOG_DIR / "04_run_training_summary.json"

RUNS_LOG_DIR.mkdir(parents=True, exist_ok=True)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)


def get_run_log_dir(scenario_name: str, seed: int) -> Path:
    return RUNS_LOG_DIR / scenario_name / f"seed_{seed}"


def log_run_event(
    event_type: str,
    scenario_name: str,
    seed: int,
    extra: dict | None = None,
) -> None:
    event = {
        "timestamp_utc": utc_now(),
        "event_type": event_type,
        "scenario": scenario_name,
        "seed": seed,
    }

    if extra:
        event.update(extra)

    append_jsonl(EVENTS_LOG_PATH, event)


print("Log global de eventos:", EVENTS_LOG_PATH)
print("Diretório de logs por run:", RUNS_LOG_DIR)

Log global de eventos: /workspace/pi-defense-exp/logs/training/04_run_training_events.jsonl
Diretório de logs por run: /workspace/pi-defense-exp/logs/training/runs


## 14. Treinamento SFT

Nesta etapa, será definida a função usada para treinar os cenários baseados em SFT:

```text
C2 — StruQ-like SFT
C4 — Instruction-Hierarchy-like SFT
```

A função recebe o nome do cenário, os arquivos de treino e validação, o diretório de saída do adaptador e a seed experimental da réplica.

Antes de iniciar cada treinamento, a função configura a seed correspondente. Isso ajuda a controlar inicialização, embaralhamento dos dados e demais fontes de aleatoriedade do processo de treinamento. Cada réplica é salva em um diretório próprio, contendo a seed no caminho, para evitar que uma execução sobrescreva outra.

O treinamento usa `SFTTrainer`, que é o componente do TRL voltado para supervised fine-tuning de modelos de linguagem.

O adaptador resultante será salvo no diretório específico da combinação cenário-seed.


In [18]:
def train_sft_scenario(
    scenario_name: str,
    train_path: Path,
    validation_path: Path,
    output_dir: Path,
    seed: int,
):
    print(f"\n=== Iniciando SFT: {scenario_name} | seed={seed} ===")
    started_at_utc = utc_now()
    start_time = time.time()

    run_log_dir = get_run_log_dir(scenario_name, seed)
    run_log_dir.mkdir(parents=True, exist_ok=True)

    log_run_event(
        event_type="run_started",
        scenario_name=scenario_name,
        seed=seed,
        extra={
            "method": "sft",
            "train_path": str(train_path),
            "validation_path": str(validation_path),
            "adapter_output_dir": str(output_dir),
            "run_log_dir": str(run_log_dir),
        },
    )

    trainer = None
    model = None

    try:
        set_experiment_seed(seed)

        output_dir.mkdir(parents=True, exist_ok=True)

        train_dataset = load_sft_dataset(train_path, tokenizer)
        eval_dataset = load_sft_dataset(validation_path, tokenizer)

        print("Train rows:", len(train_dataset))
        print("Validation rows:", len(eval_dataset))
        print("Output dir:", output_dir)
        print("Run log dir:", run_log_dir)

        model = load_base_model_for_training()

        os.environ["TENSORBOARD_LOGGING_DIR"] = str(run_log_dir)
        
        args = SFTConfig(
            output_dir=str(output_dir),
            logging_dir=str(run_log_dir),
            dataset_text_field="text",
            max_length=MAX_SEQ_LENGTH,
            packing=False,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=SFT_LEARNING_RATE,
            logging_steps=LOGGING_STEPS,
            save_steps=SAVE_STEPS,
            eval_steps=EVAL_STEPS,
            eval_strategy="steps",
            save_strategy="steps",
            save_total_limit=2,
            bf16=bf16_supported,
            fp16=not bf16_supported,
            seed=seed,
            data_seed=seed,
            report_to=[],
        )

        trainer = SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            peft_config=lora_config,
            processing_class=tokenizer,
        )

        train_result = trainer.train()
        trainer.save_model(str(output_dir))
        tokenizer.save_pretrained(str(output_dir))

        elapsed_seconds = time.time() - start_time
        finished_at_utc = utc_now()

        result = {
            "scenario": scenario_name,
            "seed": seed,
            "method": "sft",
            "status": "completed",
            "started_at_utc": started_at_utc,
            "finished_at_utc": finished_at_utc,
            "output_dir": str(output_dir),
            "run_log_dir": str(run_log_dir),
            "train_rows": len(train_dataset),
            "validation_rows": len(eval_dataset),
            "elapsed_seconds": elapsed_seconds,
            "train_metrics": train_result.metrics,
        }

        write_json(run_log_dir / "run_metadata.json", result)

        log_run_event(
            event_type="run_completed",
            scenario_name=scenario_name,
            seed=seed,
            extra={
                "method": "sft",
                "adapter_output_dir": str(output_dir),
                "run_log_dir": str(run_log_dir),
                "elapsed_seconds": elapsed_seconds,
                "train_metrics": train_result.metrics,
            },
        )

        print(f"SFT concluído para {scenario_name} com seed={seed}.")
        print("Adaptador salvo em:", output_dir)
        print("Metadados da run salvos em:", run_log_dir / "run_metadata.json")

        return result

    except Exception as error:
        elapsed_seconds = time.time() - start_time
        error_text = traceback.format_exc()

        with open(run_log_dir / "error.txt", "w", encoding="utf-8") as f:
            f.write(error_text)

        failure_metadata = {
            "scenario": scenario_name,
            "seed": seed,
            "method": "sft",
            "status": "failed",
            "started_at_utc": started_at_utc,
            "failed_at_utc": utc_now(),
            "output_dir": str(output_dir),
            "run_log_dir": str(run_log_dir),
            "elapsed_seconds": elapsed_seconds,
            "error": repr(error),
        }

        write_json(run_log_dir / "run_metadata.json", failure_metadata)

        log_run_event(
            event_type="run_failed",
            scenario_name=scenario_name,
            seed=seed,
            extra={
                "method": "sft",
                "adapter_output_dir": str(output_dir),
                "run_log_dir": str(run_log_dir),
                "elapsed_seconds": elapsed_seconds,
                "error": repr(error),
                "error_file": str(run_log_dir / "error.txt"),
            },
        )

        print(f"Falha no SFT para {scenario_name} com seed={seed}.")
        print("Erro registrado em:", run_log_dir / "error.txt")

        raise

    finally:
        if trainer is not None:
            del trainer
        if model is not None:
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 15. Treinamento DPO

Nesta etapa, será definida a função usada para treinar o cenário:

```text
C3 — SecAlign-like DPO
```

O DPO usa pares de preferência. Cada exemplo contém:

```text
prompt
chosen
rejected
```

No nosso experimento:

```text
chosen = expected_answer
rejected = attack_target
```

O objetivo é aumentar a preferência do modelo por respostas que seguem a instrução legítima e reduzir a preferência por respostas que seguem a instrução adversarial.

Assim como no SFT, o treinamento DPO também será executado uma vez para cada seed experimental. Cada réplica recebe a seed antes do treinamento e salva seu adaptador em um diretório próprio.

Essa organização é importante porque o notebook de avaliação poderá comparar os resultados por seed e depois agregar as métricas em média e desvio padrão.


In [19]:
import inspect


def make_dpo_config(output_dir: Path, run_log_dir: Path, seed: int) -> DPOConfig:
    candidate_args = {
        "output_dir": str(output_dir),
        "logging_dir": str(run_log_dir),
        "max_length": MAX_SEQ_LENGTH,
        "max_prompt_length": MAX_SEQ_LENGTH // 2,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": DPO_LEARNING_RATE,
        "logging_steps": LOGGING_STEPS,
        "save_steps": SAVE_STEPS,
        "eval_steps": EVAL_STEPS,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "save_total_limit": 2,
        "bf16": bf16_supported,
        "fp16": not bf16_supported,
        "seed": seed,
        "data_seed": seed,
        "report_to": [],
    }

    supported_args = set(inspect.signature(DPOConfig.__init__).parameters.keys())

    filtered_args = {
        key: value
        for key, value in candidate_args.items()
        if key in supported_args
    }

    ignored_args = sorted(set(candidate_args) - set(filtered_args))

    if ignored_args:
        print("Argumentos ignorados pelo DPOConfig desta versão do TRL:")
        for arg in ignored_args:
            print("-", arg)

    return DPOConfig(**filtered_args)

In [20]:
def normalize_dpo_example_boundary(example: dict) -> dict:
    prompt = example["prompt"].rstrip()
    chosen = example["chosen"].strip()
    rejected = example["rejected"].strip()

    return {
        **example,
        "prompt": f"{prompt}\n\nAnswer:",
        "chosen": f" {chosen}",
        "rejected": f" {rejected}",
    }

In [21]:
def train_dpo_scenario(
    scenario_name: str,
    train_path: Path,
    validation_path: Path,
    output_dir: Path,
    seed: int,
):
    print(f"\n=== Iniciando DPO: {scenario_name} | seed={seed} ===")
    started_at_utc = utc_now()
    start_time = time.time()

    run_log_dir = get_run_log_dir(scenario_name, seed)
    run_log_dir.mkdir(parents=True, exist_ok=True)

    log_run_event(
        event_type="run_started",
        scenario_name=scenario_name,
        seed=seed,
        extra={
            "method": "dpo",
            "train_path": str(train_path),
            "validation_path": str(validation_path),
            "adapter_output_dir": str(output_dir),
            "run_log_dir": str(run_log_dir),
        },
    )

    trainer = None
    model = None

    try:
        set_experiment_seed(seed)

        output_dir.mkdir(parents=True, exist_ok=True)

        train_dataset = load_dpo_dataset(train_path)
        eval_dataset = load_dpo_dataset(validation_path)

        train_dataset = train_dataset.map(normalize_dpo_example_boundary)
        eval_dataset = eval_dataset.map(normalize_dpo_example_boundary)

        print("Train rows:", len(train_dataset))
        print("Validation rows:", len(eval_dataset))
        print("Output dir:", output_dir)
        print("Run log dir:", run_log_dir)

        model = load_base_model_for_training()

        args = make_dpo_config(
            output_dir=output_dir,
            run_log_dir=run_log_dir,
            seed=seed,
        )

        trainer = DPOTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            peft_config=lora_config,
            processing_class=tokenizer,
        )

        train_result = trainer.train()
        trainer.save_model(str(output_dir))
        tokenizer.save_pretrained(str(output_dir))

        elapsed_seconds = time.time() - start_time
        finished_at_utc = utc_now()

        result = {
            "scenario": scenario_name,
            "seed": seed,
            "method": "dpo",
            "status": "completed",
            "started_at_utc": started_at_utc,
            "finished_at_utc": finished_at_utc,
            "output_dir": str(output_dir),
            "run_log_dir": str(run_log_dir),
            "train_rows": len(train_dataset),
            "validation_rows": len(eval_dataset),
            "elapsed_seconds": elapsed_seconds,
            "train_metrics": train_result.metrics,
        }

        write_json(run_log_dir / "run_metadata.json", result)

        log_run_event(
            event_type="run_completed",
            scenario_name=scenario_name,
            seed=seed,
            extra={
                "method": "dpo",
                "adapter_output_dir": str(output_dir),
                "run_log_dir": str(run_log_dir),
                "elapsed_seconds": elapsed_seconds,
                "train_metrics": train_result.metrics,
            },
        )

        print(f"DPO concluído para {scenario_name} com seed={seed}.")
        print("Adaptador salvo em:", output_dir)
        print("Metadados da run salvos em:", run_log_dir / "run_metadata.json")

        return result

    except Exception as error:
        elapsed_seconds = time.time() - start_time
        error_text = traceback.format_exc()

        with open(run_log_dir / "error.txt", "w", encoding="utf-8") as f:
            f.write(error_text)

        failure_metadata = {
            "scenario": scenario_name,
            "seed": seed,
            "method": "dpo",
            "status": "failed",
            "started_at_utc": started_at_utc,
            "failed_at_utc": utc_now(),
            "output_dir": str(output_dir),
            "run_log_dir": str(run_log_dir),
            "elapsed_seconds": elapsed_seconds,
            "error": repr(error),
        }

        write_json(run_log_dir / "run_metadata.json", failure_metadata)

        log_run_event(
            event_type="run_failed",
            scenario_name=scenario_name,
            seed=seed,
            extra={
                "method": "dpo",
                "adapter_output_dir": str(output_dir),
                "run_log_dir": str(run_log_dir),
                "elapsed_seconds": elapsed_seconds,
                "error": repr(error),
                "error_file": str(run_log_dir / "error.txt"),
            },
        )

        print(f"Falha no DPO para {scenario_name} com seed={seed}.")
        print("Erro registrado em:", run_log_dir / "error.txt")

        raise

    finally:
        if trainer is not None:
            del trainer
        if model is not None:
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 16. Executar treinamentos selecionados

Nesta etapa, o notebook executa os treinamentos indicados em `SCENARIOS_TO_RUN`.

Cada cenário é treinado de forma independente e sempre parte do mesmo modelo base. Além disso, cada cenário é treinado uma vez para cada seed em `EXPERIMENT_SEEDS`.

Isso produz réplicas experimentais separadas. Por exemplo, para o cenário `c2_struq_sft`, serão gerados adaptadores diferentes para:

```text
seed_42
seed_123
seed_2026
```

Os adaptadores serão salvos em diretórios específicos para cada combinação de cenário e seed:

```text
adapters/struq/seed_42/
adapters/struq/seed_123/
adapters/struq/seed_2026/

adapters/secalign/seed_42/
adapters/secalign/seed_123/
adapters/secalign/seed_2026/

adapters/ih/seed_42/
adapters/ih/seed_123/
adapters/ih/seed_2026/
```

Essa estrutura evita sobrescrita entre réplicas e mantém o rastreamento necessário para a avaliação estatística posterior.

Se ocorrer erro em algum cenário ou seed, a execução será interrompida para evitar salvar resultados parciais sem inspeção.


In [22]:
SCENARIO_OUTPUT_DIRS = {
    "c2_struq_sft": ADAPTERS_DIR / "struq",
    "c3_secalign_dpo": ADAPTERS_DIR / "secalign",
    "c4_ih_sft": ADAPTERS_DIR / "ih",
}

def get_adapter_output_dir(scenario_name: str, seed: int) -> Path:
    if scenario_name not in SCENARIO_OUTPUT_DIRS:
        raise ValueError(f"Cenário desconhecido: {scenario_name}")

    return SCENARIO_OUTPUT_DIRS[scenario_name] / f"seed_{seed}"

training_results = []

for scenario_name in SCENARIOS_TO_RUN:
    if scenario_name not in TRAINING_FILES:
        raise ValueError(f"Cenário desconhecido em SCENARIOS_TO_RUN: {scenario_name}")

    info = TRAINING_FILES[scenario_name]

    for seed in EXPERIMENT_SEEDS:
        output_dir = get_adapter_output_dir(scenario_name, seed)

        if info["method"] == "sft":
            result = train_sft_scenario(
                scenario_name=scenario_name,
                train_path=info["train"],
                validation_path=info["validation"],
                output_dir=output_dir,
                seed=seed,
            )
        elif info["method"] == "dpo":
            result = train_dpo_scenario(
                scenario_name=scenario_name,
                train_path=info["train"],
                validation_path=info["validation"],
                output_dir=output_dir,
                seed=seed,
            )
        else:
            raise ValueError(f"Método desconhecido para {scenario_name}: {info['method']}")

        training_results.append(result)

training_results_df = pd.DataFrame(training_results)
display(training_results_df)


=== Iniciando SFT: c2_struq_sft | seed=42 ===
Seed experimental configurada: 42
Train rows: 4200
Validation rows: 700
Output dir: /workspace/pi-defense-exp/adapters/struq/seed_42
Run log dir: /workspace/pi-defense-exp/logs/training/runs/c2_struq_sft/seed_42


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Adding EOS to train dataset:   0%|          | 0/4200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4200 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.583507,0.574880,0.543763,240158.000000,0.874923
200,0.583805,0.569964,0.547550,479719.000000,0.876836
263,0.550666,0.569750,0.527766,630923.000000,0.876174


SFT concluído para c4_ih_sft com seed=2026.
Adaptador salvo em: /workspace/pi-defense-exp/adapters/ih/seed_2026
Metadados da run salvos em: /workspace/pi-defense-exp/logs/training/runs/c4_ih_sft/seed_2026/run_metadata.json


,scenario,seed,method,status,started_at_utc,finished_at_utc,output_dir,run_log_dir,train_rows,validation_rows,elapsed_seconds,train_metrics
0,c2_struq_sft,42,sft,completed,2026-06-29T16:16:41.127728+00:00,2026-06-29T16:39:09.211723+00:00,/workspace/pi-defense-exp/adapters/struq/seed_42,/workspace/pi-defense-exp/logs/training/runs/c...,4200,700,1348.083979,"{'train_runtime': 1337.1004, 'train_samples_pe..."
1,c2_struq_sft,123,sft,completed,2026-06-29T16:39:09.216802+00:00,2026-06-29T17:01:31.660514+00:00,/workspace/pi-defense-exp/adapters/struq/seed_123,/workspace/pi-defense-exp/logs/training/runs/c...,4200,700,1342.443692,"{'train_runtime': 1334.5764, 'train_samples_pe..."
2,c2_struq_sft,2026,sft,completed,2026-06-29T17:01:31.664270+00:00,2026-06-29T17:24:12.371489+00:00,/workspace/pi-defense-exp/adapters/struq/seed_...,/workspace/pi-defense-exp/logs/training/runs/c...,4200,700,1360.707197,"{'train_runtime': 1352.3877, 'train_samples_pe..."
3,c3_secalign_dpo,42,dpo,completed,2026-06-29T17:24:12.374991+00:00,2026-06-29T17:38:13.417990+00:00,/workspace/pi-defense-exp/adapters/secalign/se...,/workspace/pi-defense-exp/logs/training/runs/c...,2100,350,841.042981,"{'train_runtime': 834.1819, 'train_samples_per..."
4,c3_secalign_dpo,123,dpo,completed,2026-06-29T17:38:13.426266+00:00,2026-06-29T17:52:15.520352+00:00,/workspace/pi-defense-exp/adapters/secalign/se...,/workspace/pi-defense-exp/logs/training/runs/c...,2100,350,842.094058,"{'train_runtime': 834.4206, 'train_samples_per..."
5,c3_secalign_dpo,2026,dpo,completed,2026-06-29T17:52:15.524524+00:00,2026-06-29T18:06:18.325934+00:00,/workspace/pi-defense-exp/adapters/secalign/se...,/workspace/pi-defense-exp/logs/training/runs/c...,2100,350,842.801383,"{'train_runtime': 835.728, 'train_samples_per_..."
6,c4_ih_sft,42,sft,completed,2026-06-29T18:06:18.332761+00:00,2026-06-29T18:29:06.574059+00:00,/workspace/pi-defense-exp/adapters/ih/seed_42,/workspace/pi-defense-exp/logs/training/runs/c...,4200,700,1368.241280,"{'train_runtime': 1360.2834, 'train_samples_pe..."
7,c4_ih_sft,123,sft,completed,2026-06-29T18:29:06.577373+00:00,2026-06-29T18:51:44.631098+00:00,/workspace/pi-defense-exp/adapters/ih/seed_123,/workspace/pi-defense-exp/logs/training/runs/c...,4200,700,1358.053704,"{'train_runtime': 1350.4459, 'train_samples_pe..."
8,c4_ih_sft,2026,sft,completed,2026-06-29T18:51:44.635021+00:00,2026-06-29T19:14:00.198664+00:00,/workspace/pi-defense-exp/adapters/ih/seed_2026,/workspace/pi-defense-exp/logs/training/runs/c...,4200,700,1335.563625,"{'train_runtime': 1327.8416, 'train_samples_pe..."


In [23]:
from pathlib import Path

events_path = Path("/workspace/pi-defense-exp/logs/training/04_run_training_events.jsonl")

if events_path.exists():
    lines = events_path.read_text(encoding="utf-8").strip().splitlines()
    for line in lines[-10:]:
        print(line)
else:
    print("Arquivo de eventos ainda não existe.")

{"timestamp_utc": "2026-06-29T17:38:13.426391+00:00", "event_type": "run_started", "scenario": "c3_secalign_dpo", "seed": 123, "method": "dpo", "train_path": "/workspace/pi-defense-exp/data/views/secalign/train_dpo.jsonl", "validation_path": "/workspace/pi-defense-exp/data/views/secalign/validation_dpo.jsonl", "adapter_output_dir": "/workspace/pi-defense-exp/adapters/secalign/seed_123", "run_log_dir": "/workspace/pi-defense-exp/logs/training/runs/c3_secalign_dpo/seed_123"}
{"timestamp_utc": "2026-06-29T17:52:15.520982+00:00", "event_type": "run_completed", "scenario": "c3_secalign_dpo", "seed": 123, "method": "dpo", "adapter_output_dir": "/workspace/pi-defense-exp/adapters/secalign/seed_123", "run_log_dir": "/workspace/pi-defense-exp/logs/training/runs/c3_secalign_dpo/seed_123", "elapsed_seconds": 842.094057559967, "train_metrics": {"train_runtime": 834.4206, "train_samples_per_second": 2.517, "train_steps_per_second": 0.158, "total_flos": 1.714527594283008e+16, "train_loss": 0.0591864

## 17. Conferência dos adaptadores gerados

Nesta etapa, o notebook confere se os diretórios de saída dos adaptadores foram criados e se contêm arquivos após o treinamento.

Como agora cada cenário possui múltiplas réplicas, a conferência também considera a seed. Assim, a verificação não procura apenas por um adaptador de `c2_struq_sft`, mas por um adaptador para cada combinação cenário-seed executada.

Essa verificação não garante sozinha a qualidade dos adaptadores, mas ajuda a detectar falhas simples, como diretórios vazios, salvamento incompleto ou execução ausente.

A qualidade dos adaptadores será avaliada posteriormente no notebook de avaliação, usando os mesmos arquivos comuns de teste para todos os cenários e agregando os resultados por seed.


In [24]:
adapter_status = []

for scenario_name in SCENARIOS_TO_RUN:
    for seed in EXPERIMENT_SEEDS:
        output_dir = get_adapter_output_dir(scenario_name, seed)
        files = sorted([str(path.relative_to(output_dir)) for path in output_dir.rglob("*") if path.is_file()]) if output_dir.exists() else []

        adapter_status.append({
            "scenario": scenario_name,
            "seed": seed,
            "output_dir": str(output_dir),
            "exists": output_dir.exists(),
            "num_files": len(files),
            "sample_files": files[:10],
        })

adapter_status_df = pd.DataFrame(adapter_status)
display(adapter_status_df)

for row in adapter_status:
    if not row["exists"] or row["num_files"] == 0:
        raise RuntimeError(
            f"Adaptador ausente ou vazio para {row['scenario']} "
            f"com seed={row['seed']}: {row['output_dir']}"
        )


,scenario,seed,output_dir,exists,num_files,sample_files
0,c2_struq_sft,42,/workspace/pi-defense-exp/adapters/struq/seed_42,True,29,"[README.md, adapter_config.json, adapter_model..."
1,c2_struq_sft,123,/workspace/pi-defense-exp/adapters/struq/seed_123,True,29,"[README.md, adapter_config.json, adapter_model..."
2,c2_struq_sft,2026,/workspace/pi-defense-exp/adapters/struq/seed_...,True,29,"[README.md, adapter_config.json, adapter_model..."
3,c3_secalign_dpo,42,/workspace/pi-defense-exp/adapters/secalign/se...,True,29,"[README.md, adapter_config.json, adapter_model..."
4,c3_secalign_dpo,123,/workspace/pi-defense-exp/adapters/secalign/se...,True,29,"[README.md, adapter_config.json, adapter_model..."
5,c3_secalign_dpo,2026,/workspace/pi-defense-exp/adapters/secalign/se...,True,29,"[README.md, adapter_config.json, adapter_model..."
6,c4_ih_sft,42,/workspace/pi-defense-exp/adapters/ih/seed_42,True,29,"[README.md, adapter_config.json, adapter_model..."
7,c4_ih_sft,123,/workspace/pi-defense-exp/adapters/ih/seed_123,True,29,"[README.md, adapter_config.json, adapter_model..."
8,c4_ih_sft,2026,/workspace/pi-defense-exp/adapters/ih/seed_2026,True,29,"[README.md, adapter_config.json, adapter_model..."


## 18. Gerar log resumido do treinamento

Nesta etapa, será salvo um resumo final da execução do treinamento.

Esse resumo não substitui os logs incrementais gerados durante cada run. Os logs incrementais são mais importantes para rastrear a execução em tempo real, porque registram início, conclusão e falha assim que cada evento acontece.

O objetivo desta etapa é consolidar, ao final do notebook, as principais informações em dois formatos:

```text
logs/training/04_run_training.log
logs/training/04_run_training_summary.json
```

O arquivo `.log` é uma versão simples para inspeção rápida. O arquivo `.json` é mais adequado para leitura automatizada por scripts ou notebooks posteriores.

Esses arquivos registram o modelo base, a seed do dataset, as seeds experimentais de treinamento, a precisão usada, a GPU, os resultados concluídos e os caminhos dos logs incrementais.

Se o notebook for interrompido antes desta etapa, os arquivos finais podem não ser gerados. Nesse caso, a principal fonte de rastreamento continua sendo:

```text
logs/training/04_run_training_events.jsonl
logs/training/runs/
```

In [25]:
log_path = LOG_DIR / "04_run_training.log"

summary_log = {
    "notebook": "04_run_training",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": BASE_MODEL_ID,
    "dataset_seed": DATASET_SEED,
    "experiment_seeds": EXPERIMENT_SEEDS,
    "precision": precision_name,
    "gpu": device_name,
    "events_log_path": str(EVENTS_LOG_PATH),
    "runs_log_dir": str(RUNS_LOG_DIR),
    "training_results": training_results,
}

write_json(SUMMARY_LOG_JSON_PATH, summary_log)

with open(log_path, "w", encoding="utf-8") as f:
    f.write("04_run_training\n")
    f.write(f"created_at_utc: {summary_log['created_at_utc']}\n")
    f.write(f"base_model: {BASE_MODEL_ID}\n")
    f.write(f"dataset_seed: {DATASET_SEED}\n")
    f.write(f"experiment_seeds: {EXPERIMENT_SEEDS}\n")
    f.write(f"precision: {precision_name}\n")
    f.write(f"gpu: {device_name}\n")
    f.write(f"events_log_path: {EVENTS_LOG_PATH}\n")
    f.write(f"runs_log_dir: {RUNS_LOG_DIR}\n")
    f.write(f"summary_log_json_path: {SUMMARY_LOG_JSON_PATH}\n")
    f.write("\ntraining_results:\n")
    for result in training_results:
        f.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")

print("Log resumido salvo em:", log_path)
print("Resumo JSON salvo em:", SUMMARY_LOG_JSON_PATH)
print("Log incremental de eventos:", EVENTS_LOG_PATH)
print("Diretório de logs por run:", RUNS_LOG_DIR)

Log resumido salvo em: /workspace/pi-defense-exp/logs/training/04_run_training.log
Resumo JSON salvo em: /workspace/pi-defense-exp/logs/training/04_run_training_summary.json
Log incremental de eventos: /workspace/pi-defense-exp/logs/training/04_run_training_events.jsonl
Diretório de logs por run: /workspace/pi-defense-exp/logs/training/runs


## 19. Gerar manifesto do treinamento

Nesta etapa final, será gerado o manifesto do notebook de treinamento.

O manifesto registra o estado da execução, os cenários treinados, o modelo base, as configurações LoRA/QLoRA, os arquivos de treino usados, os diretórios dos adaptadores, as seeds experimentais, os caminhos dos logs incrementais e informações do ambiente.

Serão criados dois arquivos:

```text
manifests/training/04_run_training_manifest.json
manifests/training/04_run_training_manifest.md
```

O arquivo `.json` é voltado para leitura automatizada por notebooks posteriores. O arquivo `.md` é voltado para inspeção manual.

A inclusão das seeds no manifesto é essencial para a etapa de avaliação. O notebook de avaliação precisa saber quais réplicas foram produzidas para cada cenário, onde cada adaptador está salvo e qual seed corresponde a cada resultado.

Esse manifesto será usado para identificar quais adaptadores estão disponíveis e para permitir que as métricas finais sejam agregadas por cenário, reportando média, desvio padrão e demais estatísticas sobre as três seeds experimentais.

Os caminhos dos logs também são registrados no manifesto para que o notebook de avaliação ou uma inspeção posterior consigam localizar rapidamente o histórico de cada run.


In [26]:
manifest_json_path = MANIFEST_DIR / "04_run_training_manifest.json"
manifest_md_path = MANIFEST_DIR / "04_run_training_manifest.md"

total_expected_adapter_runs = len(SCENARIOS_TO_RUN) * len(EXPERIMENT_SEEDS)

manifest = {
    "notebook": "04_run_training",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "base_model_id": BASE_MODEL_ID,
    "dataset_seed": DATASET_SEED,
    "experiment_seeds": EXPERIMENT_SEEDS,
    "replicates_per_trained_scenario": len(EXPERIMENT_SEEDS),
    "total_expected_adapter_runs": total_expected_adapter_runs,
    "expected_python": str(EXPECTED_PYTHON),
    "current_python": str(CURRENT_PYTHON),
    "system": {
        "platform": platform.platform(),
        "python_version": sys.version,
    },
    "gpu": {
        "device_name": device_name,
        "total_memory_gb": total_memory_gb,
        "cuda_version": torch.version.cuda,
        "bf16_supported": bf16_supported,
        "precision": precision_name,
    },
    "hyperparameters": training_hyperparameters,
    "lora": lora_plan,
    "training_files": {
        scenario: {
            "method": info["method"],
            "train": str(info["train"]),
            "validation": str(info["validation"]),
            "expected_train_rows": info["expected_train_rows"],
            "expected_validation_rows": info["expected_validation_rows"],
        }
        for scenario, info in TRAINING_FILES.items()
    },
    "scenarios_run": SCENARIOS_TO_RUN,
    "training_results": training_results,
    "adapter_status": adapter_status,
    "log_path": str(log_path),
    "summary_log_json_path": str(SUMMARY_LOG_JSON_PATH),
    "events_log_path": str(EVENTS_LOG_PATH),
    "runs_log_dir": str(RUNS_LOG_DIR),
    "notes": [
        "C0 and C1 do not train adapters and are not executed in this notebook.",
        "C2 and C4 use SFT over clean + attacked seen examples.",
        "C3 uses DPO over attacked seen examples only.",
        "All trained scenarios start from the same base model checkpoint.",
        "Each trained scenario is executed once for each seed in EXPERIMENT_SEEDS.",
        "Adapter directories include seed-specific subdirectories to avoid overwriting experimental replicates.",
        "Training runs are logged incrementally in a global JSONL event log.",
        "Each scenario-seed pair has an individual run log directory.",
    ],
}

with open(manifest_json_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)

results_table_md = "\n".join([
    "| Scenario | Seed | Method | Train rows | Validation rows | Adapter dir | Elapsed seconds |",
    "|---|---:|---|---:|---:|---|---:|",
    *[
        f"| `{result['scenario']}` | {result['seed']} | `{result['method']}` | {result['train_rows']} | {result['validation_rows']} | `{result['output_dir']}` | {result['elapsed_seconds']:.2f} |"
        for result in training_results
    ]
])

adapter_table_md = "\n".join([
    "| Scenario | Seed | Exists | Number of files | Output dir |",
    "|---|---:|---:|---:|---|",
    *[
        f"| `{row['scenario']}` | {row['seed']} | {row['exists']} | {row['num_files']} | `{row['output_dir']}` |"
        for row in adapter_status
    ]
])

manifest_md = f"""# Manifesto de treinamento — Notebook 04

## Identificação

- Notebook: `04_run_training`
- Gerado em UTC: `{manifest['created_at_utc']}`
- Diretório raiz do projeto: `{PROJECT_ROOT}`
- Modelo base: `{BASE_MODEL_ID}`

## Seeds

- Seed do dataset: `{DATASET_SEED}`
- Seeds experimentais de treinamento: `{EXPERIMENT_SEEDS}`
- Réplicas por cenário treinável: `{len(EXPERIMENT_SEEDS)}`
- Total esperado de execuções de adaptadores: `{total_expected_adapter_runs}`

A seed do dataset identifica a base experimental criada no notebook `02_dataset_creation.ipynb`. Ela deve permanecer fixa para preservar os mesmos splits, amostras e ataques.

As seeds experimentais de treinamento representam réplicas independentes para os cenários C2, C3 e C4. Cada adaptador salvo neste notebook deve estar associado a uma seed específica.

## Ambiente

- Python atual: `{CURRENT_PYTHON}`
- Python esperado: `{EXPECTED_PYTHON}`
- Plataforma: `{platform.platform()}`

## GPU

- GPU: `{device_name}`
- Memória total aproximada: `{total_memory_gb:.2f} GB`
- CUDA no PyTorch: `{torch.version.cuda}`
- Precisão usada: `{precision_name}`

## Cenários executados

{', '.join([f'`{scenario}`' for scenario in SCENARIOS_TO_RUN])}

## Resultados de treinamento

{results_table_md}

## Adaptadores

{adapter_table_md}

## Configuração LoRA/QLoRA

- r: `{lora_config.r}`
- lora_alpha: `{lora_config.lora_alpha}`
- lora_dropout: `{lora_config.lora_dropout}`
- target_modules: `{lora_config.target_modules}`
- quantização: `4-bit nf4`
- compute dtype: `{precision_name}`

## Logs

- Log resumido textual: `{log_path}`
- Log resumido JSON: `{SUMMARY_LOG_JSON_PATH}`
- Log incremental de eventos: `{EVENTS_LOG_PATH}`
- Diretório de logs por run: `{RUNS_LOG_DIR}`

O log incremental de eventos registra início, conclusão e falha de cada combinação cenário-seed. Os diretórios individuais de run armazenam metadados específicos e, em caso de erro, um arquivo `error.txt` com o traceback.

## Observações

- C0 e C1 não treinam adaptadores e não são executados neste notebook.
- C2 e C4 usam SFT com exemplos limpos e atacados vistos.
- C3 usa DPO apenas com exemplos atacados vistos.
- Todos os cenários treinados partem do mesmo checkpoint base.
- Cada cenário treinável é executado uma vez para cada seed experimental.
- Os adaptadores são salvos em subdiretórios específicos por seed.
- A qualidade dos adaptadores será medida posteriormente no notebook de avaliação.
"""

with open(manifest_md_path, "w", encoding="utf-8") as f:
    f.write(manifest_md)

print("Manifesto JSON criado em:", manifest_json_path)
print("Manifesto Markdown criado em:", manifest_md_path)

Manifesto JSON criado em: /workspace/pi-defense-exp/manifests/training/04_run_training_manifest.json
Manifesto Markdown criado em: /workspace/pi-defense-exp/manifests/training/04_run_training_manifest.md
